In [ ]:
import numpy as np
import pandas as pd 
import pickle
from pathlib import Path
from astropy.timeseries import LombScargle
from scipy.stats import skew, kurtosis, shapiro
from utils.preprocessing import fourier_features, stetson_K, fourier_fit

DATA_DIR = Path("/home/admin/main/ucsd-phys-139-final/data")
FEATURES_PATH = DATA_DIR / "features.csv"
LABELED_FEATURES_PATH = FEATURES_PATH.with_name(FEATURES_PATH.stem + "_labeled.csv")

GAIA_MATCH = DATA_DIR / "gaia_crossmatch/gaia_crossmatch.csv"
ASASSN_MATCH = DATA_DIR / "asassn/asassn_crossmatch.csv"

In [ ]:
def compute_features(light_curves):
    rows = []
    print(f"Processing {len(light_curves)} light curves...")

    for idx, item in enumerate(light_curves):
        if isinstance(item, (list, tuple)) and len(item) == 3:
            star_id, t, f = item
        else:
            t, f = item
            star_id = idx

        mask = np.isfinite(t) & np.isfinite(f)
        t, f = t[mask], f[mask]

        try:
            freq, power = LombScargle(t, f).autopower()
            best_period = 1 / freq[np.argmax(power)]
        except Exception:
            best_period = np.nan

        if len(f) > 0:
            Q1 = np.percentile(f, 25)
            Q3 = np.percentile(f, 75)
            Q31 = Q3 - Q1
            Std = np.std(f)
            gamma1 = skew(f)
            gamma2 = kurtosis(f, fisher=True)
            W, _ = shapiro(f)
            K = stetson_K(f)
            R21, R31, phi21, phi31, Amp = fourier_features(best_period, t, f)
        else:
            Q31 = Std = gamma1 = gamma2 = W = K = R21 = R31 = phi21 = phi31 = Amp = np.nan

        rows.append({
            "star_id": star_id,
            "period": best_period,
            "Q31": Q31,
            "Amp": Amp,
            "W": W,
            "K": K,
            "Std": Std,
            "gamma1": gamma1,
            "gamma2": gamma2,
            "R21": R21,
            "R31": R31,
            "phi21": phi21,
            "phi31": phi31
        })

        if idx % 1000 == 0:
            print(f"Processed {idx}/{len(light_curves)}")

    df = pd.DataFrame(rows)
    ordered_cols = [
        "star_id", "period", "Q31", "Amp", "W", "K", "Std",
        "gamma1", "gamma2", "R21", "R31", "phi21", "phi31"
    ]
    return df[ordered_cols]

In [ ]:
if not FEATURES_PATH.exists():
    raise FileNotFoundError(f"Features file not found: {FEATURES_PATH}")

df_features = pd.read_csv(FEATURES_PATH)
print(f"Loaded {len(df_features)} rows from {FEATURES_PATH}")

In [ ]:
has_filenames = False
if 'light_curves' in locals() and len(light_curves) > 0:
    item = light_curves[0]
    if len(item) == 3:
        print("Pickle contains (id, t, f). Using ID for matching.")
        has_filenames = True
    else:
        print("Pickle contains only (t, f). WARNING: Cannot reliably match to external catalogs without IDs.")
        print("Creating dummy labels for demonstration logic.")

gaia_df = pd.read_csv(GAIA_MATCH) if GAIA_MATCH.exists() else pd.DataFrame()
asassn_df = pd.read_csv(ASASSN_MATCH) if ASASSN_MATCH.exists() else pd.DataFrame()

print(f"Loaded {len(gaia_df)} Gaia matches")
print(f"Loaded {len(asassn_df)} ASASSN matches")

In [ ]:
df_features['label'] = 'Unknown'

if has_filenames:
    if not asassn_df.empty:
        asassn_map = dict(zip(asassn_df['TESS file name'], asassn_df['ASASSN Class']))
        df_features['label'] = df_features['star_id'].map(asassn_map).fillna(df_features['label'])
    
    if not gaia_df.empty and 'Gaia_Class' in gaia_df.columns:
        gaia_map = dict(zip(gaia_df['TESS_Filename'], gaia_df['Gaia_Class']))
        mask_unknown = df_features['label'] == 'Unknown'
        new_labels = df_features.loc[mask_unknown, 'star_id'].map(gaia_map)
        df_features.loc[mask_unknown, 'label'] = new_labels.fillna('Unknown')

df_features.to_csv(LABELED_FEATURES_PATH, index=False)
print(f"Saved labeled features to {LABELED_FEATURES_PATH}")
print("Sample of labeled data:")
print(df_features[df_features['label'] != 'Unknown'].head())